# Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import matplotlib as mpl

plt.style.use('dark_background')
mpl.rcParams['legend.frameon'] = False

In [ ]:
plt.rcParams.update({
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "lines.markersize": 7,
})

In [ ]:
# define folder to output generated figures
path_figures = Path('./figures')
path_figures.mkdir(exist_ok=True)

# define figure parameters
def save_figure(filename):
    plt.savefig(f"{path_figures}/{filename}.svg", transparent=True)

# Introduction

This is a problem that appears in the book "Gödel, Escher, Bach" by Douglas Hofstadter. This book is on my to-read list, but the following problem was presented to me by a friend and without context (in the book, this exercise illustrates a deeper idea).

>Can you characterize the following set of integers (or its negative space)?
>
> ```1 3 7 12 18 26 35 45 56 69```

# A series based on the difference of differences

In [ ]:
# the number of numbers I want to generate
N = 12

In [ ]:
def generate_dd(N):
    r_size = 2 # this is the size of the repeat; I initialize it just before 2 1 1
    r_left = 0 # this is how many numbers are left to fill the repeat of size r_size
    list_dds = []
    
    for i in range(N):
        # if it is the end of the repeat:
        if r_left == 0:
            dd = 2           # add a 2 to the list_dds
            r_size += 1      # increase the repeat size
            r_left = r_size  # reset the r_left counter
        else:
            dd = 1           # add a 1 to the list_dds
            
        list_dds.append(dd)  # update list_dds  
        r_left -= 1          # update r_left counter
        
    return list_dds

In [ ]:
# the number of numbers I want to generate
N = 12

list_dds = generate_dd(N)
print(len(list_dds))
print(list_dds)

In [ ]:
def generate_diffs(list_dds):
    # we initialize the list with a 2
    temp_list = [2] + list_dds
    list_diffs = np.cumsum(temp_list).tolist()
    return list_diffs

In [ ]:
list_diffs = generate_diffs(list_dds)
print(len(list_diffs))
print(list_diffs)

In [ ]:
def generate_series(list_diffs):
    temp_list = [1] + list_diffs
    list_series = np.cumsum(temp_list).tolist()
    return list_series

In [ ]:
list_series = generate_series(list_diffs)
print(len(list_series))
print(list_series)

Indeed, this looks like the series of numbers that we were asked to characterize

```1 3 7 12 18 26 35 45 56 69```

In [ ]:
# sanity checks (do not let paranoia settle in)
assert np.array_equal(np.array(list_diffs), np.diff(list_series))
assert np.array_equal(np.array(list_dds), np.diff(list_diffs))

In [ ]:
# we can check that, indeed, the sets of list_diffs, and list_series do not intersect!
set(list_series).intersection(set(list_diffs))

In [ ]:
# the number of numbers I want to generate
N = 200

In [ ]:
list_dds = generate_dd(N)
list_diffs = generate_diffs(list_dds)
list_series = generate_series(list_diffs)

In [ ]:
# sanity checks (do not let paranoia settle in)
assert np.array_equal(np.array(list_diffs), np.diff(list_series))
assert np.array_equal(np.array(list_dds), np.diff(list_diffs))

In [ ]:
# we can check that, indeed, the sets of list_diffs, and list_series do not intersect!
set(list_series).intersection(set(list_diffs))

# Hofstadter Figure-Figure sequence

The Hofstadter Figure-Figure (R and S) sequences are a pair of complementary integer sequences defined as follows

$$ R(1) = 1; S(1) = 2 $$
$$ R(n) = R(n-1) + S(n-1), n > 1 $$

with the sequence $S(n)$ defined as a strictly increasing series of positive integers not present in $R(n)$

In [ ]:
def compute_Hofstadter_sequence(N=10):
    # initialization
    R = [1]
    S = [2]

    # extension
    for i in range(N):
        # extend R
        new_R = R[i] + S[i]
        R.append(new_R)

        # extend S
        candidate_S = S[i] + 1
        while True:
            if candidate_S not in R:
                new_S = candidate_S
                break
            else:
                candidate_S += 1
                
        S.append(new_S)
        
    # finalization
    return R, S

In [ ]:
R, S = compute_Hofstadter_sequence(10)

In [ ]:
print(R)
print(S)

In [ ]:
# sanity check (still cannot let paranoia settle in!)
assert np.array_equal(np.diff(R), np.array(S[:-1]))

In [ ]:
# we can check that, the R and S sets do not intersect!
set(R).intersection(set(S))

And, finally, we will check that this property holds for largers segments of these sets

In [ ]:
R, S = compute_Hofstadter_sequence(10000)

In [ ]:
# we can check that, the R and S sets do not intersect!
set(R).intersection(set(S))

## Visualization

In [ ]:
R, S = compute_Hofstadter_sequence(10)

In [ ]:
# create indices to remember which set each value came from
mat_R = np.vstack((R, np.zeros_like(R))).T
mat_S = np.vstack((S, np.ones_like(R))).T
# stack the two matrices
mat_data = np.vstack([mat_R, mat_S])

print(mat_R.shape)
print(mat_S.shape)
print(mat_data.shape)

In [ ]:
# sort them by value
mat_sorting_mask = np.argsort(mat_data[:,0])
sorted_mat_data = mat_data[mat_sorting_mask]

In [ ]:
sorted_mat_data[:10,:]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2))
colors = ['C0' if i == 0 else 'C1' for i in sorted_mat_data[:, 1]]
ax.scatter(x=sorted_mat_data[:,0], y = sorted_mat_data[:,1], c=colors)
plt.plot(sorted_mat_data[:,0],sorted_mat_data[:,1], color='gray', linewidth=1, zorder=5)
ax.set_ylim([-0.5,1.5])
ax.set_yticks(ticks=[0,1], labels=['R', 'S'])
plt.tight_layout()
save_figure('shadow_series_22')

In [ ]:
R, S = compute_Hofstadter_sequence(1000)

In [ ]:
# create indices to remember which set each value came from
mat_R = np.vstack((R, np.zeros_like(R))).T
mat_S = np.vstack((S, np.ones_like(R))).T
# stack the two matrices
mat_data = np.vstack([mat_R, mat_S])

print(mat_R.shape)
print(mat_S.shape)
print(mat_data.shape)

In [ ]:
# sort them by value
mat_sorting_mask = np.argsort(mat_data[:,0])
sorted_mat_data = mat_data[mat_sorting_mask]

# only keep values under a threshold
sorted_mat_data = sorted_mat_data[sorted_mat_data[:,0] < 100]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2))
colors = ['C0' if i == 0 else 'C1' for i in sorted_mat_data[:, 1]]
ax.scatter(x=sorted_mat_data[:,0], y = sorted_mat_data[:,1], c=colors)
plt.plot(sorted_mat_data[:,0],sorted_mat_data[:,1], color='gray', linewidth=1, zorder=5)
ax.set_ylim([-0.5,1.5])
ax.set_yticks(ticks=[0,1], labels=['R', 'S'])
plt.tight_layout()
save_figure('shadow_series_100')

In [ ]:
# I remove the first two members of the combined series, and map the 0-1 values to to 2-1
list_21_pattern = (1-sorted_mat_data[:,1]+1).tolist()[2:]
print(list_21_pattern)

In [ ]:
str_21_pattern = ''.join(f"\n2 " if val == 2
                         else "1 "
                         for val in list_21_pattern)
print(str_21_pattern)

In [ ]:
R, S = compute_Hofstadter_sequence(1000)

mat_R = np.vstack((R, np.zeros_like(R))).T
mat_S = np.vstack((S, np.ones_like(R))).T
mat_data = np.vstack([mat_R, mat_S])
mat_sorting_mask = np.argsort(mat_data[:,0])
sorted_mat_data = mat_data[mat_sorting_mask]
sorted_mat_data = sorted_mat_data[sorted_mat_data[:,0] < 900]

In [ ]:
list_21_pattern = (1-sorted_mat_data[:,1]+1).tolist()[2:]
str_21_pattern = ''.join(f"\n2 " if val == 2
                         else "1 "
                         for val in list_21_pattern)
print(str_21_pattern)

For 3 rounds, the `2 1 1 1` pattern adds `1`, then `11`

For 4 rounds, the pattern adds `1`, and then `11`

For 5 rounds, the patterns adds `1` and then `11`

Then... then it skips to 7 rounds, not 6. Oh well.

# Conclusions

This was fun. Even though my initial motivation to develop this notebook was rooted in a wrong premise, I think the exercise was useful. One could make a phylosophical point about the fact that if someone asks

>Can you characterize the following set of integers (or its negative space)?
>
> ```1 3 7 12 18 26 35 45 56 69```

there are multiple correct answers, and, from a certain point of view, I was not wrong, but rather, I was fed incomplete data! I am not salty at all, even though I will mention, in no way to defend myself, that, from my research, several people on the internet had the same idea as I.

Screw you Douglas Hofstadter, I am looking forward to read "Gödel, Escher, Bach".